# Notebook 05 — Node Embeddings

## Purpose
Test whether random-walk-based embeddings (Node2Vec) add discriminative 
signal for fraud detection, given notebook 04's finding that PageRank 
(also walk-based) was fully redundant with in-degree due to the fraud 
subgraph's sparse, star-dominated structure.

## Panel decision
Run a structural diagnostic on the FULL graph first (not fraud subgraph) 
to check whether meaningful multi-hop random walks are feasible before 
committing to a full training run. Evaluate embeddings against the 
in-degree baseline honestly, whatever the result.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle

with open('../data/processed/transaction_graph.pkl', 'rb') as f:
    G = pickle.load(f)

# Diagnostic: can meaningful multi-hop walks even happen on this graph?
G_undirected = G.to_undirected()

# Component size distribution on the FULL graph
components = list(nx.connected_components(G_undirected))
component_sizes = [len(c) for c in components]

print(f"Total components: {len(components):,}")
print(f"Largest component size: {max(component_sizes):,}")
print(f"Largest component as % of all nodes: {max(component_sizes)/G.number_of_nodes()*100:.2f}%")
print(f"Number of components with size > 10: {sum(1 for s in component_sizes if s > 10)}")
print(f"Number of components with size > 100: {sum(1 for s in component_sizes if s > 100)}")
print(f"Median component size: {np.median(component_sizes)}")

## Finding: Node2Vec is structurally infeasible on this graph

A pre-training diagnostic on the full transaction graph (3,277,489 nodes) 
found:
- 507,096 disconnected components
- Largest component: 90 nodes (0.00% of total graph)
- Zero components exceed 100 nodes
- Median component size: 4

Random-walk-based embedding methods (Node2Vec, DeepWalk) rely on walks of 
typical length 40-80 steps to capture multi-hop neighbourhood context. With 
a hard ceiling of 90 nodes in the largest possible connected piece anywhere 
in the graph, such walks are structurally infeasible almost everywhere — 
this is the same underlying limitation that made PageRank redundant with 
in-degree in notebook 04, now confirmed to extend across the entire graph, 
not just the fraud subgraph.

**Decision:** do not train Node2Vec. The diagnostic — costing seconds — 
avoided a training run that would very likely have reproduced the null 
result already found for PageRank, for the identical structural reason. 
This is documented as a deliberate, evidence-based scoping decision rather 
than a skipped step.

## Why this graph is so fragmented
This is itself a genuine finding worth stating plainly: PaySim's account 
population is dominated by one-time participants (88% of nodes have degree 
1, from notebook 01) rather than a smaller set of frequently-reused 
accounts. Real-world financial networks — where the same merchants, 
employers, and utilities receive from thousands of different customers 
repeatedly — would likely show a very different, much more connected 
structure. This is a concrete, evidence-backed limitation of PaySim as a 
proxy for real transaction networks, directly relevant to the "does this 
generalise beyond a synthetic dataset" question raised throughout this 
project.

## What would be needed for embeddings to be viable
Noted as future work: a real-world dataset with repeat-customer structure 
(e.g. a merchant/consumer network rather than one-off P2P transfers) would 
have the multi-hop connectivity Node2Vec requires. Alternatively, a 
neighbourhood-aggregation embedding approach (e.g. single-hop GraphSAGE-style 
aggregation, which only requires 1-hop neighbours rather than long walks) 
would sidestep this specific limitation and is a better-suited technique 
for graphs this sparse — proposed as future work rather than built here, 
consistent with project scope.